# 1. Data Loading

- **Reproducibility**: Setting up robust paths ensures the notebook works across different machines and environments
- **Quick Validation**: Initial checks help identify data issues early in the analysis pipeline
- **Understanding Schema**: Creating a data dictionary provides immediate insight into column structure and data quality
- **Error Prevention**: Proper path handling prevents common file not found errors
- **Scalability**: Modular approach makes it easy to add more data sources later
- **Documentation**: Clear markdown cells explain the purpose and methodology for future reference
- **Quality Assurance**: Systematic checks ensure data integrity before proceeding to analysis

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Set up project paths robustly
PROJECT_ROOT = Path.cwd().parent  # Go up from /notebooks to repo root
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
VISUALS_DIR = PROJECT_ROOT / "visuals"

# Create missing directories
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
VISUALS_DIR.mkdir(parents=True, exist_ok=True)

print("Project paths resolved:")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"RAW_DIR: {RAW_DIR}")
print(f"PROCESSED_DIR: {PROCESSED_DIR}")
print(f"VISUALS_DIR: {VISUALS_DIR}")

In [ ]:
# List available data files
if not RAW_DIR.exists():
    raise FileNotFoundError(f"Raw data directory not found: {RAW_DIR}")

# Find CSV and Excel files
csv_files = list(RAW_DIR.glob("*.csv"))
excel_files = list(RAW_DIR.glob("*.xlsx"))
all_files = csv_files + excel_files

if not all_files:
    raise FileNotFoundError(f"No CSV or Excel files found in {RAW_DIR}")

print(f"Found {len(all_files)} data file(s):")
for i, file in enumerate(all_files, 1):
    print(f"  {i}. {file.name}")

# Select the first CSV by default (easy to change)
if csv_files:
    DATA_PATH = csv_files[0]
else:
    DATA_PATH = excel_files[0]

print(f"\nChosen file: {DATA_PATH.name}")
print(f"Full path: {DATA_PATH}")

In [ ]:
# Load the dataset
if DATA_PATH.suffix.lower() == '.csv':
    df = pd.read_csv(DATA_PATH, low_memory=False)
elif DATA_PATH.suffix.lower() in ['.xlsx', '.xls']:
    df = pd.read_excel(DATA_PATH)
else:
    raise ValueError(f"Unsupported file format: {DATA_PATH.suffix}")

print(f"Dataset loaded successfully!")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

## Quick Sanity Checks

In [ ]:
print("=== First 5 rows ===")
display(df.head(5))

print("\n=== Last 5 rows ===")
display(df.tail(5))

if len(df) >= 5:
    print("\n=== Random sample of 5 rows ===")
    display(df.sample(5, random_state=42))

print("\n=== DataFrame Info ===")
df.info()

print("\n=== Descriptive Statistics ===")
# Limit output if there are many columns
max_cols_to_show = 30
if df.shape[1] <= max_cols_to_show:
    display(df.describe(include="all").T)
else:
    print(f"Showing first {max_cols_to_show} columns (out of {df.shape[1]}):")
    display(df.describe(include="all").T.head(max_cols_to_show))

## Data Quality Snapshot

In [ ]:
# Missing values analysis
missing_counts = df.isnull().sum()
missing_percentages = (missing_counts / len(df)) * 100
missing_df = pd.DataFrame({
    'missing_count': missing_counts,
    'missing_pct': missing_percentages
}).sort_values('missing_count', ascending=False)

print("=== Missing Values ===")
display(missing_df[missing_df['missing_count'] > 0])

# Duplicate rows
duplicate_count = df.duplicated().sum()
print(f"\n=== Duplicate Rows ===")
print(f"Number of duplicate rows: {duplicate_count:,}")
print(f"Percentage of duplicates: {(duplicate_count / len(df)) * 100:.2f}%")

# Unique values per column
unique_counts = df.nunique()
print(f"\n=== Unique Values per Column ===")
# Show top 25 columns by uniqueness if there are many columns
if len(unique_counts) > 25:
    print("Showing top 25 columns by number of unique values:")
    display(unique_counts.sort_values(ascending=False).head(25))
else:
    display(unique_counts.sort_values(ascending=False))

# Memory usage
memory_mb = df.memory_usage(deep=True).sum() / 1024**2
print(f"\n=== Memory Usage ===")
print(f"Total memory usage: {memory_mb:.2f} MB")

In [ ]:
# Create data dictionary
schema_data = []
for col in df.columns:
    col_data = {
        'column_name': col,
        'dtype': str(df[col].dtype),
        'non_null_count': df[col].count(),
        'null_count': df[col].isnull().sum(),
        'null_pct': (df[col].isnull().sum() / len(df)) * 100,
        'n_unique': df[col].nunique()
    }
    schema_data.append(col_data)

schema_df = pd.DataFrame(schema_data)

print("=== Data Dictionary ===")
display(schema_df.sort_values('null_pct', ascending=False))

## Notes / Next Steps

- **Next notebook section** will handle cleaning (missing values strategy, type conversions, normalization)
- **We will save a processed dataset** to `data/processed` later after cleaning transformations
- **Reproducibility**: All paths are set up relative to project root, making the notebook portable
- **Version control**: Consider committing the raw data dictionary to track schema changes over time
- **Data quality**: Review the missing values and duplicates before proceeding to cleaning steps